# DisasterSeverity – BanglaBERT v2  
**Targeting 0.80+ weighted F1** (baseline v1: 0.73)

| # | Improvement | Expected gain |
|---|-------------|---------------|
| 1 | Category-aware input text | +3–5 % |
| 2 | Label smoothing (0.1) | +0.5–1 % |
| 3 | Cosine LR + warm-up | +0.5–1 % |
| 4 | R-Drop regularisation | +1–2 % |
| 5 | FGM adversarial training | +1–2 % |
| 6 | Pseudo-labelling (0.90 threshold) | +0.5–1 % |
| – | Pre-computed class weights (not per-step) | (speed) |

In [1]:
# ── Cell 1: Imports & Seed ─────────────────────────────────────────────────
import os, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from scipy.special import softmax as scipy_softmax
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    set_seed,
)
from datasets import Dataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import zipfile

warnings.filterwarnings("ignore")

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    set_seed(seed)

seed_everything(42)

In [2]:
# ── Cell 2: Configuration ──────────────────────────────────────────────────
class CFG:
    # ── Model
    # Swap to "csebuetnlp/banglabert_large" for extra ~2% at the cost of
    # slower training and batch_size=8 + gradient_accumulation_steps=2.
    # xlm-roberta-large is another strong option.
    model_name = "csebuetnlp/banglabert"
    base_path  = "/kaggle/input/competitions/datathon-iiuc-cse-fest-2026/DisasterSeverity/"

    # ── Tokenisation
    # 95th-pct of texts is ~44 words; with BanglaBERT subword factor (~2.3×)
    # + category prefix (~5 tokens) → ~106 tokens at 95th pct. 128 is safe.
    max_len = 128

    # ── Training
    epochs       = 5        # v1 used 4
    batch_size   = 16
    lr           = 2e-5
    warmup_ratio = 0.10     # 10 % of steps used for cosine warm-up
    weight_decay = 0.01

    # ── Loss
    label_smoothing = 0.10  # prevents overconfident logits; 0.05–0.15 is safe range

    # ── Regularisation
    use_rdrop   = True      # R-Drop: symmetric KL between two dropout-stochastic passes
    rdrop_alpha = 0.50      # weight for KL divergence term
    use_fgm     = True      # FGM: adversarial perturbation of word embeddings
    fgm_epsilon = 0.50

    # ── Cross-Validation
    n_folds = 5
    seed    = 42

    # ── Pseudo-Labelling (second-pass semi-supervised training)
    use_pseudo       = True
    pseudo_threshold = 0.90   # only samples the model is ≥90 % confident about
    pseudo_epochs    = 3
    pseudo_lr        = 1e-5   # lower LR for fine-tuning on pseudo data

label_map         = {"Minimal": 0, "Mild": 1, "Moderate": 2, "Severe": 3, "Catastrophic": 4}
reverse_label_map = {v: k for k, v in label_map.items()}

In [3]:
# ── Cell 3: Data Loading — IMPROVEMENT #1: Category-Aware Input ───────────
#
# Why this works:
#   EDA shows Non Disaster → always Minimal (the existing rule),
#   but other categories also have strong skews:
#     Tropical Storm → 47 % Mild, only 0.4 % Catastrophic
#     Wildfire       → 44 % Severe
#     Earthquake     → 44 % Moderate
#   Feeding the category as a text prefix lets the model learn ALL these
#   patterns *from gradients* instead of only from the hard-coded rule.
# ────────────────────────────────────────────────────────────────────────────
print("Loading data...")
train = pd.read_csv(f"{CFG.base_path}train.csv")
test  = pd.read_csv(f"{CFG.base_path}test.csv")
val   = pd.read_csv(f"{CFG.base_path}validation.csv")

# Combine all labelled data for cross-validation
train = pd.concat([train, val]).reset_index(drop=True)

# ★ Key change: prepend the disaster category to the context text
train["text"] = train["category"] + ": " + train["context"].fillna("")
test["text"]  = test["category"]  + ": " + test["context"].fillna("")

train["label_id"] = train["label"].map(label_map)

# Stratified K-Fold
skf = StratifiedKFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)
train["fold"] = -1
for fold, (_, val_idx) in enumerate(skf.split(train, train["label_id"])):
    train.loc[val_idx, "fold"] = fold

tokenizer = AutoTokenizer.from_pretrained(CFG.model_name)

def tokenize_fn(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=CFG.max_len,
    )

test_dataset   = Dataset.from_pandas(test[["text"]])
tokenized_test = test_dataset.map(tokenize_fn, batched=True)

# ★ Pre-compute class weights ONCE (v1 recomputed them every training step)
class_counts  = np.bincount(train["label_id"].values, minlength=5).astype(float)
CLASS_WEIGHTS = (len(train) / (5 * class_counts)).tolist()
print("Class weights:", {k: round(CLASS_WEIGHTS[v], 3) for k, v in label_map.items()})
print(f"Train: {len(train)} samples | Test: {len(test)} samples")

Loading data...


config.json:   0%|          | 0.00/586 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/790 [00:00<?, ? examples/s]

Class weights: {'Minimal': 1.171, 'Mild': 0.985, 'Moderate': 0.659, 'Severe': 0.8, 'Catastrophic': 2.751}
Train: 3590 samples | Test: 790 samples


In [4]:
# ── Cell 4: FGM + AdvancedTrainer (R-Drop · FGM · label-smoothing) ─────────

# ── IMPROVEMENT #5: Fast Gradient Method ────────────────────────────────────
class FGM:
    """
    Fast Gradient Method – perturbs word-embedding weights in the direction
    of the loss gradient to force the model to be robust to small changes.
    Typically +1–2 % on Bangla classification tasks.
    """
    def __init__(self, model):
        self.model  = model
        self.backup = {}

    def attack(self, epsilon=0.5, emb_name="embeddings"):
        for name, param in self.model.named_parameters():
            if param.requires_grad and emb_name in name and param.grad is not None:
                self.backup[name] = param.data.clone()
                norm = torch.norm(param.grad)
                if norm != 0 and not torch.isnan(norm):
                    param.data.add_(epsilon * param.grad / norm)

    def restore(self, emb_name="embeddings"):
        for name, param in self.model.named_parameters():
            if param.requires_grad and emb_name in name and name in self.backup:
                param.data = self.backup[name]
        self.backup = {}


# ── Metrics (unchanged – judges' exact metric) ───────────────────────────────
def compute_metrics(pred):
    labels = pred.label_ids
    preds  = pred.predictions.argmax(-1)
    _, _, f1, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1}


# ── IMPROVEMENTS #2, #3, #4: AdvancedTrainer ─────────────────────────────────
class AdvancedTrainer(Trainer):
    """
    Stacks four training improvements on top of vanilla Trainer:
      #2 – Balanced CE loss (class weights, pre-computed)
      #3 – Label smoothing (prevents overconfident logits)
      #4 – R-Drop (symmetric KL between two stochastic forward passes)
      #5 – FGM adversarial embedding perturbation
    """

    def _ce(self, logits, labels):
        """Weighted cross-entropy with label smoothing."""
        wt = torch.tensor(CLASS_WEIGHTS, dtype=torch.float32).to(logits.device)
        return nn.CrossEntropyLoss(weight=wt, label_smoothing=CFG.label_smoothing)(
            logits, labels
        )

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")

        # R-Drop only fires during training (model.training=True);
        # disabled during eval/predict so we don't waste two forward passes.
        # Also disabled during the FGM adversarial step (_fgm_mode flag).
        if model.training and CFG.use_rdrop and not getattr(self, "_fgm_mode", False):
            # ── Two forward passes with different dropout masks ──────────
            out1 = model(**inputs)
            out2 = model(**inputs)

            ce = (self._ce(out1.logits, labels) + self._ce(out2.logits, labels)) / 2

            # Symmetric KL divergence
            p1  = F.softmax(out1.logits, dim=-1)
            p2  = F.softmax(out2.logits, dim=-1)
            kl  = (
                F.kl_div(out1.logits.log_softmax(-1), p2, reduction="batchmean") +
                F.kl_div(out2.logits.log_softmax(-1), p1, reduction="batchmean")
            ) / 2

            loss = ce + CFG.rdrop_alpha * kl
            return (loss, out1) if return_outputs else loss

        else:
            out  = model(**inputs)
            loss = self._ce(out.logits, labels)
            return (loss, out) if return_outputs else loss

    def training_step(self, model, inputs, num_items_in_batch=None, **kwargs):
        model.train()
        inputs = self._prepare_inputs(inputs)

        # Save labels before compute_loss pops them from the dict
        labels = inputs["labels"].clone()

        # ── Normal forward + backward ────────────────────────────────────
        with self.compute_loss_context_manager():
            loss = self.compute_loss(model, inputs)

        if self.args.gradient_accumulation_steps > 1:
            loss = loss / self.args.gradient_accumulation_steps

        self.accelerator.backward(loss)

        # ── FGM adversarial backward ─────────────────────────────────────
        if CFG.use_fgm:
            fgm = FGM(model)
            fgm.attack(epsilon=CFG.fgm_epsilon)

            inputs["labels"] = labels           # restore (was popped above)
            self._fgm_mode = True               # skip R-Drop for this pass
            with self.compute_loss_context_manager():
                loss_adv = self.compute_loss(model, inputs)
            self._fgm_mode = False

            if self.args.gradient_accumulation_steps > 1:
                loss_adv = loss_adv / self.args.gradient_accumulation_steps
            self.accelerator.backward(loss_adv)
            fgm.restore()

        return loss.detach()

In [5]:
# ── Cell 5: 5-Fold Training — IMPROVEMENT #6: Cosine LR + More Epochs ────
test_predictions = []

for fold in range(CFG.n_folds):
    print(f"\n{'='*22} FOLD {fold+1}/{CFG.n_folds} {'='*22}")

    trn_df = train[train["fold"] != fold].reset_index(drop=True)
    val_df = train[train["fold"] == fold].reset_index(drop=True)

    tok_trn = (
        Dataset.from_pandas(trn_df[["text", "label_id"]].rename(columns={"label_id": "label"}))
        .map(tokenize_fn, batched=True)
    )
    tok_val = (
        Dataset.from_pandas(val_df[["text", "label_id"]].rename(columns={"label_id": "label"}))
        .map(tokenize_fn, batched=True)
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        CFG.model_name, num_labels=5
    )

    # ★ Key changes vs v1:
    #   – lr_scheduler_type = "cosine"  (was linear default)
    #   – warmup_ratio = 0.10           (was no warmup)
    #   – num_train_epochs = 5          (was 4)
    args = TrainingArguments(
        output_dir                  = f"/kaggle/working/fold_{fold}",
        eval_strategy               = "epoch",
        save_strategy               = "epoch",
        learning_rate               = CFG.lr,
        per_device_train_batch_size = CFG.batch_size,
        per_device_eval_batch_size  = CFG.batch_size,
        num_train_epochs            = CFG.epochs,
        warmup_ratio                = CFG.warmup_ratio,   # ← new
        lr_scheduler_type           = "cosine",           # ← new
        weight_decay                = CFG.weight_decay,
        load_best_model_at_end      = True,
        metric_for_best_model       = "f1",
        greater_is_better           = True,
        report_to                   = "none",
        save_total_limit            = 1,
    )

    trainer = AdvancedTrainer(
        model           = model,
        args            = args,
        train_dataset   = tok_trn,
        eval_dataset    = tok_val,
        compute_metrics = compute_metrics,
    )

    trainer.train()
    print(f"Predicting test set for fold {fold + 1}...")
    preds = trainer.predict(tokenized_test).predictions
    test_predictions.append(preds)


====================== FOLD 1/5 ======================


Map:   0%|          | 0/2872 [00:00<?, ? examples/s]

Map:   0%|          | 0/718 [00:00<?, ? examples/s]

pytorch_model.bin:   0%|          | 0.00/443M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['electra.embeddings.LayerNorm.weight', 'electra.embeddings.LayerNorm.bias', 'electra.encoder.layer.0.attention.output.LayerNorm.weight', 'electra.encoder.layer.0.attention.output.LayerNorm.bias', 'electra.encoder.layer.0.output.LayerNorm.weight', 'electra.encoder.layer.0.output.LayerNorm.bias', 'electra.encoder.layer.1.attention.output.LayerNorm.weight', 'electra.encoder.layer.1.attention.output.LayerNorm.bias', 'electra.encoder.layer.1.output.LayerNorm.weight', 'electra.encoder.layer.1.output.LayerNorm.bias', 'electra.encoder.layer.2.attention.output.LayerNorm.weight', 'electra.encoder.layer.2.attention.output.LayerNorm.bias', 'electra.encoder.layer.2.output.LayerNorm.weight', 'electra.encoder.layer.2.output.LayerNorm.bias', 'electra.encoder.layer.3.attention.output.LayerNorm.weight', 'electra.encoder.layer.3.attention.output.LayerNorm.bias', 'electra.encoder.layer.3.output.LayerNorm.weight', 'electra.encoder.layer.3.output.Laye

Predicting test set for fold 5...


In [6]:
# ── Cell 6: Ensemble + Rule-Based Post-Processing ─────────────────────────
print("\nEnsembling 5-fold predictions...")
final_logits = np.mean(test_predictions, axis=0)
final_preds  = np.argmax(final_logits, axis=-1)

test["label"] = [reverse_label_map[p] for p in final_preds]

# Rule: Non Disaster is 100% Minimal in training data (verified 450/450)
# This overrides even very confident model predictions.
test.loc[test["category"] == "Non Disaster", "label"] = "Minimal"
print("Rule applied: Non Disaster → Minimal")
print("\nPrediction distribution:")
print(test["label"].value_counts())


Ensembling 5-fold predictions...
Rule applied: Non Disaster → Minimal

Prediction distribution:
label
Mild            184
Severe          174
Moderate        170
Catastrophic    151
Minimal         111
Name: count, dtype: int64


In [7]:
# ── Cell 7: IMPROVEMENT #7 — Pseudo-Labelling ────────────────────────────
#
# Strategy:
#   1. Convert 5-fold ensemble logits to probabilities.
#   2. Keep only test samples where max-prob ≥ pseudo_threshold (0.90).
#   3. Train a final model on (all labelled train) + (high-confidence pseudo)
#      with a lower LR and fewer epochs.
#   4. Blend: 70 % CV ensemble + 30 % pseudo model.
# ────────────────────────────────────────────────────────────────────────────
if CFG.use_pseudo:
    print("\n── Pseudo-Labelling ──")
    probs     = scipy_softmax(final_logits, axis=-1)
    max_probs = probs.max(axis=-1)

    confident = max_probs >= CFG.pseudo_threshold
    print(f"High-confidence test samples: {confident.sum()}/{len(test)} "
          f"(threshold={CFG.pseudo_threshold})")

    if confident.sum() > 50:   # only proceed if we have a meaningful number
        pseudo_df = test[confident].copy()
        pseudo_df["label_id"] = final_preds[confident]

        full_train = pd.concat(
            [train[["text", "label_id"]], pseudo_df[["text", "label_id"]]],
            ignore_index=True,
        )
        print(f"Expanded train size: {len(train)} → {len(full_train)} samples")

        tok_full = (
            Dataset.from_pandas(full_train.rename(columns={"label_id": "label"}))
            .map(tokenize_fn, batched=True)
        )

        pseudo_model = AutoModelForSequenceClassification.from_pretrained(
            CFG.model_name, num_labels=5
        )
        pseudo_args = TrainingArguments(
            output_dir                  = "/kaggle/working/pseudo",
            num_train_epochs            = CFG.pseudo_epochs,
            per_device_train_batch_size = CFG.batch_size,
            per_device_eval_batch_size  = CFG.batch_size,
            learning_rate               = CFG.pseudo_lr,
            warmup_ratio                = CFG.warmup_ratio,
            lr_scheduler_type           = "cosine",
            weight_decay                = CFG.weight_decay,
            save_strategy               = "no",
            report_to                   = "none",
        )
        pseudo_trainer = AdvancedTrainer(
            model           = pseudo_model,
            args            = pseudo_args,
            train_dataset   = tok_full,
            compute_metrics = compute_metrics,
        )
        pseudo_trainer.train()

        pseudo_logits = pseudo_trainer.predict(tokenized_test).predictions

        # Blend: give more weight to the robust 5-fold ensemble
        blended      = 0.70 * final_logits + 0.30 * pseudo_logits
        final_preds  = np.argmax(blended, axis=-1)
        test["label"] = [reverse_label_map[p] for p in final_preds]

        # Re-apply the hard rule after blending
        test.loc[test["category"] == "Non Disaster", "label"] = "Minimal"
        print("Blended predictions updated (70% CV + 30% pseudo).")
    else:
        print("Too few high-confidence samples; skipping pseudo-labelling.")


── Pseudo-Labelling ──
High-confidence test samples: 100/790 (threshold=0.9)
Expanded train size: 3590 → 3690 samples


Map:   0%|          | 0/3690 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

ElectraForSequenceClassification LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     | 
--------------------------------------------------+------------+-
electra.embeddings.position_ids                   | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
classifier.dense.bias                             | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoi

Step,Training Loss


Blended predictions updated (70% CV + 30% pseudo).


In [8]:
# ── Cell 8: Save Submission ────────────────────────────────────────────────
submission = test[["image_id", "label"]]
submission.to_csv("submission.csv", index=False)

with zipfile.ZipFile("submission.zip", "w") as z:
    z.write("submission.csv", arcname="submission.csv")

print("\n✅ 'submission.zip' is ready for upload.")
print("\nFinal prediction distribution:")
print(submission["label"].value_counts())
print()
print(submission.head(10))


✅ 'submission.zip' is ready for upload.

Final prediction distribution:
label
Mild            197
Moderate        165
Severe          162
Catastrophic    161
Minimal         105
Name: count, dtype: int64

         image_id         label
0  landslides_804  Catastrophic
1  landslides_805      Moderate
2  landslides_806          Mild
3  landslides_807          Mild
4  landslides_808          Mild
5  landslides_809          Mild
6  landslides_810          Mild
7  landslides_811          Mild
8  landslides_812          Mild
9  landslides_813        Severe


## Next steps if you want to push above 0.82

| Option | Change | Notes |
|--------|--------|-------|
| **Larger model** | `CFG.model_name = "csebuetnlp/banglabert_large"` | Set `batch_size=8`, `gradient_accumulation_steps=2`. Likely +2–3 %. |
| **XLM-R large** | `"xlm-roberta-large"` | Very strong multilingual baseline. May outperform BanglaBERT on this dataset. |
| **Multi-model ensemble** | Average logits from BanglaBERT + XLM-R | Training both adds GPU hours but typically +1–2 %. |
| **Larger max_len** | `max_len=200` | Helps the ~5 % of texts > 128 subword tokens. |
| **More folds** | `n_folds=10` | Reduces variance; each model trains on 90 % of data. |